# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [80]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

In [81]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [82]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="wings-uhm.github.io/Web/")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [83]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    date = data['date'].strftime("%b %d, %Y")
    description = f"<strong/>{title} </strong>- {date}<br />{venue}; {location} "

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

<strong/>Conference Proceeding talk 3 on Relevant Topic in Your Field </strong>- Mar 01, 2014<br />Testing Institute of America 2014 Annual Conference; Los Angeles, CA, USA  Los Angeles, Los Angeles County, California, United States
<strong/>Intelligent Open RAN: AI-Driven Integrated Sensing and Communication (ISAC) for Trusted Next Generation Wireless Networks </strong>- Apr 17, 2025<br />College of Engineering, University of Hawaiʻi at Mānoa; Holmes Hall, 2540 Dole St, Honolulu, HI 96822  Holmes Hall, 2540, Dole Street, Moiliili, McCully, East Honolulu, Honolulu, Honolulu County, Hawaii, 96822, United States
<strong/>Enhancing Security and Privacy in Distributed Wireless Networks Through Physical Layer Techniques </strong>- Feb 01, 2025<br />Stevens Institute of Technology; Stevens Institute of Technology, Hoboken, NJ, USA  Stevens Institute of Technology, 1, Hudson Street, Uptown, Hoboken, Hudson County, New Jersey, 07030, United States
<strong/>Integrated Sensing and Communication 

In [84]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'